# Optional extra: double descent

In Monday's lecture we saw that test error can go **down again** when a network has far more parameters than it needs to fit the training data. This is *double descent*. This notebook reproduces the experiment from the lecture so you can play with it.

**The setup.** We take 20 noisy samples of $\sin(x)$ and fit an MLP with one hidden layer of $K$ ReLUs:
$$g(x)=\sum_{k=1}^K v_k\,\mathrm{ReLU}(w_k x+b_k).$$
To keep things simple, the hidden layer weights $w_k, b_k$ are chosen at random and **frozen**. Only the output weights $v_k$ are learnt. This means fitting the network is just linear regression (from last week!), with the $K$ hidden unit outputs as the features.

When $K$ is larger than the number of training points, there are infinitely many choices of $\mathbf{v}$ that fit the data exactly. `torch.linalg.lstsq` then returns the one with the **smallest norm** $\|\mathbf{v}\|$.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

torch.set_default_dtype(torch.float64)   # least squares is numerically delicate near the interpolation threshold

## Data, features and fitting

In [ ]:
n, noise = 20, 0.1                        # number of training points, noise standard deviation

def make_data(seed):
    g = torch.Generator().manual_seed(seed)
    # training inputs spread evenly(ish) over [0, 2*pi]
    x = (torch.arange(n) + 0.2 + 0.6 * torch.rand(n, generator=g)) * 2 * np.pi / n
    y = torch.sin(x) + noise * torch.randn(n, generator=g)
    return x, y

x_test = torch.linspace(0, 2 * np.pi, 400)
y_test = torch.sin(x_test)

def random_hidden_layer(K, seed):
    # Random, frozen hidden layer: unit k has a "kink" somewhere in [-1, 2*pi+1]
    g = torch.Generator().manual_seed(seed + 1000)
    w = torch.randn(K, generator=g)
    kinks = -1 + (2 * np.pi + 2) * torch.rand(K, generator=g)
    return w, -w * kinks

def hidden(x, w, b):
    # Hidden layer outputs, shape (len(x), K)
    return torch.relu(x[:, None] * w + b) / np.sqrt(len(w))

def fit(x, y, w, b):
    # Output weights by least squares. Driver 'gelsd' returns the minimum-norm solution
    # when there are more hidden units than training points.
    return torch.linalg.lstsq(hidden(x, w, b), y[:, None], driver='gelsd').solution[:, 0]

## The experiment

For each number of hidden units $K$ we fit many networks (different datasets and different random hidden layers) and record the median training and test error. This takes about a minute.

In [ ]:
Ks = np.unique(np.round(np.geomspace(4, 10000, 36)).astype(int))
n_datasets, n_draws = 15, 30

train_err, test_err = [], []
for K in Ks:
    tr, te = [], []
    for d in range(n_datasets):
        x, y = make_data(d)
        for s in range(n_draws):
            w, b = random_hidden_layer(K, s)
            v = fit(x, y, w, b)
            tr.append(torch.mean((hidden(x, w, b) @ v - y) ** 2).item())
            te.append(torch.mean((hidden(x_test, w, b) @ v - y_test) ** 2).item())
    train_err.append(np.median(tr))
    test_err.append(np.median(te))

plt.figure(figsize=(8, 4.5))
plt.semilogx(Ks, train_err, label='training error', color='grey')
plt.semilogx(Ks, test_err, label='test error', lw=2)
plt.ylim(0, 1.3 * max(np.array(test_err)[Ks >= 10]))
plt.xlabel('number of hidden units K')
plt.ylabel('mean squared error')
plt.legend()
plt.show()

## What do the fitted functions look like?

In [ ]:
x, y = make_data(3)
fig, axes = plt.subplots(1, 4, figsize=(16, 3.2))
for ax, K in zip(axes, [4, 14, 32, 10000]):
    w, b = random_hidden_layer(K, 0)
    v = fit(x, y, w, b)
    ax.plot(x_test, y_test, '--', color='lightgrey')
    ax.plot(x_test, hidden(x_test, w, b) @ v)
    ax.plot(x, y, 'k.', ms=8)
    ax.set_ylim(-1.6, 1.6)
    ax.set_title(f'K = {K},  ||v|| = {v.norm():.1f}')
plt.show()

## Things to try

1. **Noise.** Set `noise = 0` and re-run. What happens to the peak? Why might the peak be caused by fitting the noise?
2. **Number of training points.** Change `n` to 10 or 40. Where does the peak move to?
3. **Regularisation.** Add L2 regularisation (ridge regression, as last week) to `fit`: solve $(\Phi^\top\Phi+\lambda I)\mathbf{v}=\Phi^\top\mathbf{y}$ with a small $\lambda$. What happens to the peak?
4. **Gradient descent and the smallest-norm solution.** Fit the output weights with gradient descent instead of `lstsq`, starting from $\mathbf{v}=\mathbf{0}$, for a network with many more hidden units than training points. The cell below does this. Every gradient is a combination of the rows of the hidden-layer matrix $H$, so gradient descent never moves in a direction the training data can't "see". If you run it to convergence it therefore ends up at the minimum-norm solution. Check that the part of $\mathbf{v}$ outside the row space of $H$ stays (numerically) zero. Then look at how slowly $\mathbf{v}$ approaches the `lstsq` solution, and what happens to the test error along the way. This is a preview of *early stopping* in Week 5.

In [ ]:
x, y = make_data(3)
w, b = random_hidden_layer(1000, 0)
H = hidden(x, w, b)
v_lstsq = fit(x, y, w, b)
row_space = torch.linalg.pinv(H) @ H          # projects onto the row space of H

v = torch.zeros(1000, requires_grad=True)
optimiser = torch.optim.SGD([v], lr=0.3)
for step in range(1, 50001):
    loss = torch.mean((H @ v - y) ** 2)
    optimiser.zero_grad()
    loss.backward()
    optimiser.step()
    if step in [10, 100, 1000, 10000, 50000]:
        vd = v.detach()
        test = torch.mean((hidden(x_test, w, b) @ vd - y_test) ** 2).item()
        print(f'step {step:6d}: train {loss.item():.4f}  test {test:.4f}  ||v|| {vd.norm():6.2f}  '
              f'distance to lstsq {(vd - v_lstsq).norm():6.2f}  outside row space {(vd - row_space @ vd).norm():.1e}')

test = torch.mean((hidden(x_test, w, b) @ v_lstsq - y_test) ** 2).item()
print(f'lstsq (min norm): train 0.0000  test {test:.4f}  ||v|| {v_lstsq.norm():6.2f}')